# Silver → Gold

Modelo dimensional (Star Schema) para análise dos preços.

- `dim_produto` e `dim_calendario`: dimensões simples, carregadas com MERGE
- `dim_posto`: histórico de bandeira e nome de cada posto (SCD Tipo 2)
- `fato_precos`: um registro por posto, produto e dia, ligado à versão do posto válida na data da coleta
- `vw_precos`: fato com as dimensões e o preço corrigido pelo IPCA

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalogo = "preco_na_bomba"
silver = f"{catalogo}.silver"
gold = f"{catalogo}.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")

df_precos = spark.table(f"{silver}.tb_precos")

## 1. dim_produto

Chave substituta gerada pelo Delta (`IDENTITY`). O MERGE só insere produtos que ainda não existem.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.dim_produto (
        sk_produto BIGINT GENERATED ALWAYS AS IDENTITY,
        produto STRING,
        unidade_medida STRING
    )
""")

df_precos.select("produto", "unidade_medida").distinct().createOrReplaceTempView("produtos_silver")

spark.sql(f"""
    MERGE INTO {gold}.dim_produto AS t
    USING produtos_silver AS s
    ON t.produto = s.produto
    WHEN NOT MATCHED THEN INSERT (produto, unidade_medida) VALUES (s.produto, s.unidade_medida)
""")

display(spark.table(f"{gold}.dim_produto").orderBy("sk_produto"))

sk_produto,produto,unidade_medida
1,Gasolina Aditivada,R$ / litro
2,GNV,R$ / m³
3,Diesel S10,R$ / litro
4,Diesel,R$ / litro
5,Etanol,R$ / litro
6,Gasolina,R$ / litro


## 2. dim_calendario

Um dia por linha, do primeiro ao último ano com coletas. A chave `sk_data` é a própria data no formato `yyyyMMdd`.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.dim_calendario (
        sk_data INT,
        data DATE,
        ano INT,
        semestre INT,
        trimestre INT,
        mes INT,
        ano_mes STRING,
        dia_semana INT,
        fim_de_semana BOOLEAN
    )
""")

inicio, fim = df_precos.agg(F.min("data_coleta"), F.max("data_coleta")).first()

(
    spark.sql(f"""
        SELECT explode(sequence(
            trunc(DATE'{inicio}', 'year'),
            make_date(year(DATE'{fim}'), 12, 31),
            INTERVAL 1 DAY
        )) AS data
    """)
    .select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("sk_data"),
        "data",
        F.year("data").alias("ano"),
        F.when(F.month("data") <= 6, 1).otherwise(2).alias("semestre"),
        F.quarter("data").alias("trimestre"),
        F.month("data").alias("mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
        F.dayofweek("data").alias("dia_semana"),
        F.dayofweek("data").isin(1, 7).alias("fim_de_semana"),
    )
    .createOrReplaceTempView("calendario_novo")
)

spark.sql(f"""
    MERGE INTO {gold}.dim_calendario AS t
    USING calendario_novo AS s
    ON t.sk_data = s.sk_data
    WHEN NOT MATCHED THEN INSERT *
""")

display(spark.sql(f"SELECT MIN(data), MAX(data), COUNT(*) AS dias FROM {gold}.dim_calendario"))

MIN(data),MAX(data),dias
2024-01-01,2025-12-31,731


## 3. dim_posto (SCD Tipo 2)

Quando um posto troca de bandeira ou de nome, a versão antiga é fechada e uma nova é aberta.

- **Observação:** um registro por posto e dia, com os atributos daquele dia
- **Versão nova:** começa no primeiro dia em que a bandeira ou o nome muda em relação à observação anterior
- `data_inicio` e `data_fim`: a versão vale de `data_inicio` (inclusive) até `data_fim` (exclusive); a versão atual tem `data_fim = 9999-12-31` e `ativo = true`
- `hash_atributos`: resumo da bandeira e do nome, usado para detectar a mudança
- **MERGE:** as versões são recalculadas a partir de todo o histórico da Silver, e o MERGE fecha a versão que ganhou sucessora e insere as novas; rodar de novo não cria versões repetidas

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold}.dim_posto (
        sk_posto BIGINT GENERATED ALWAYS AS IDENTITY,
        cnpj STRING,
        revenda STRING,
        bandeira STRING,
        regiao STRING,
        uf STRING,
        municipio STRING,
        bairro STRING,
        cep STRING,
        hash_atributos STRING,
        data_inicio DATE,
        data_fim DATE,
        ativo BOOLEAN
    )
""")

atributos = ["revenda", "bandeira", "regiao", "uf", "municipio", "bairro", "cep"]
rastreados = ["revenda", "bandeira"]
fim_aberto = F.lit(date(9999, 12, 31))
por_posto = Window.partitionBy("cnpj").orderBy("data_coleta")

observacoes = (
    df_precos
    .groupBy("cnpj", "data_coleta")
    .agg(*[F.max(c).alias(c) for c in atributos])
    .withColumn(
        "hash_atributos",
        F.sha2(F.concat_ws("|", *[F.coalesce(F.col(c), F.lit("")) for c in rastreados]), 256),
    )
)

versoes = (
    observacoes
    .withColumn("hash_anterior", F.lag("hash_atributos").over(por_posto))
    .filter(~F.col("hash_atributos").eqNullSafe(F.col("hash_anterior")))
    .withColumn("data_inicio", F.col("data_coleta"))
    .withColumn("data_fim", F.coalesce(F.lead("data_coleta").over(por_posto), fim_aberto))
    .withColumn("ativo", F.col("data_fim") == fim_aberto)
    .select("cnpj", *atributos, "hash_atributos", "data_inicio", "data_fim", "ativo")
)

versoes.createOrReplaceTempView("versoes_posto")

spark.sql(f"""
    MERGE INTO {gold}.dim_posto AS t
    USING versoes_posto AS s
    ON t.cnpj = s.cnpj AND t.data_inicio = s.data_inicio
    WHEN MATCHED AND t.data_fim <> s.data_fim THEN
        UPDATE SET t.data_fim = s.data_fim, t.ativo = s.ativo
    WHEN NOT MATCHED THEN
        INSERT (cnpj, revenda, bandeira, regiao, uf, municipio, bairro, cep, hash_atributos, data_inicio, data_fim, ativo)
        VALUES (s.cnpj, s.revenda, s.bandeira, s.regiao, s.uf, s.municipio, s.bairro, s.cep, s.hash_atributos, s.data_inicio, s.data_fim, s.ativo)
""")

ativos_repetidos = spark.sql(f"""
    SELECT cnpj FROM {gold}.dim_posto WHERE ativo GROUP BY cnpj HAVING COUNT(*) > 1
""").count()
assert ativos_repetidos == 0, f"{ativos_repetidos} postos com mais de uma versão ativa"

In [0]:
display(spark.sql(f"""
    SELECT COUNT(DISTINCT cnpj) AS postos,
           COUNT(*) AS versoes,
           COUNT(*) - COUNT(DISTINCT cnpj) AS mudancas
    FROM {gold}.dim_posto
"""))

display(spark.sql(f"""
    WITH exemplo AS (
        SELECT cnpj FROM {gold}.dim_posto
        GROUP BY cnpj HAVING COUNT(DISTINCT bandeira) > 1
        ORDER BY cnpj LIMIT 1
    )
    SELECT sk_posto, cnpj, revenda, bandeira, data_inicio, data_fim, ativo
    FROM {gold}.dim_posto
    WHERE cnpj IN (SELECT cnpj FROM exemplo)
    ORDER BY data_inicio
"""))

postos,versoes,mudancas
12389,14000,1611


sk_posto,cnpj,revenda,bandeira,data_inicio,data_fim,ativo
26,00005892000113,M L Souza & Cia Ltda,TOTALENERGIES,2024-03-26,2025-08-25,false
37,00005892000113,M L Souza & Cia Ltda,NEXTA,2025-08-25,9999-12-31,true
